# 🏆 Churn Prediction: Telco Production Pipeline (V6.0 - TELCO EDITION)
**Dataset**: [blastchar/telco-customer-churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)
**Focus**: High-quality real-world features + Version Lock + Full Analytics.

In [ ]:
# 1. Setup & Auth
!pip uninstall umap-learn hdbscan -y -q
!pip install opendatasets mlflow xgboost shap python-dotenv seaborn dagshub scikit-learn==1.5.1 -q

import sklearn; print(f"✅ Environment Ready: {sklearn.__version__}")
import os, pandas as pd, numpy as np, mlflow, mlflow.sklearn, matplotlib.pyplot as plt, seaborn as sns, shap, dagshub
import opendatasets as od
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

mlflow.sklearn.autolog(log_models=False)
dagshub.init(repo_owner="nhannhb92", repo_name="msa24-ddm501-group6-final-project", mlflow=True)

In [ ]:
# 2. Telco Specific Cleaning
def clean_telco_data(df):
    df.columns = [col.lower() for col in df.columns]
    # Remove ID
    df = df.drop(columns=[c for c in ['customerid'] if c in df.columns])
    
    # Totalcharges has spaces for new customers
    df['totalcharges'] = pd.to_numeric(df['totalcharges'], errors='coerce')
    
    # Convert Churn to 1/0
    if 'churn' in df.columns:
        df['churn'] = df['churn'].map({'Yes': 1, 'No': 0})
    
    return df.dropna()

if not os.path.exists("telco-customer-churn"):
    od.download("https://www.kaggle.com/datasets/blastchar/telco-customer-churn")

df_raw = clean_telco_data(pd.read_csv("telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"))
df_train, df_test = train_test_split(df_raw, test_size=0.2, random_state=42, stratify=df_raw['churn'])

print("\n--- 📊 TELCO DATA PROFILE ---")
print(f"Total Records: {len(df_raw)}")
print(f"Churn Rate: {df_raw['churn'].mean():.2%}")
print(f"Features: {list(df_train.columns)}")

In [ ]:
# 3. Full Visual EDA (Telco Edition)
plt.figure(figsize=(18, 5))
plt.subplot(1, 4, 1); sns.countplot(x='churn', data=df_train); plt.title("Class Distribution")
plt.subplot(1, 4, 2); sns.histplot(x='tenure', hue='churn', data=df_train, kde=True); plt.title("Tenure vs Churn")
plt.subplot(1, 4, 3); sns.boxplot(x='churn', y='monthlycharges', data=df_train); plt.title("Monthly Charges")
plt.subplot(1, 4, 4); sns.countplot(x='contract', hue='churn', data=df_train); plt.title("Contract Type")
plt.tight_layout(); plt.show()

In [ ]:
# 4. Training Engine (Telco Optimized + Responsible AI)
def run_telco_experiment(df_tr, df_te, model_type):
    X_tr, y_tr = df_tr.drop(columns=['churn']), df_tr['churn']
    X_te, y_te = df_te.drop(columns=['churn']), df_te['churn']
    
    num_f = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    cat_f = X_tr.select_dtypes(include=['object']).columns.tolist()
    
    pre = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_f),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_f)
    ])
    
    if model_type == "xgboost":
        pos_w = (y_tr == 0).sum() / (y_tr == 1).sum()
        clf = XGBClassifier(scale_pos_weight=pos_w, n_estimators=100, max_depth=5, eval_metric='logloss')
        m_p = {'clf__max_depth': [3, 5, 7]}
    else:
        clf = LogisticRegression(max_iter=2000, class_weight='balanced')
        m_p = {'clf__C': [0.1, 1.0, 10.0]}
        
    mlflow.set_experiment("Churn_Telco_Production")
    with mlflow.start_run(run_name=f"NB_TELCO_{model_type.upper()}"):
        pipe = Pipeline([('pre', pre), ('clf', clf)])
        search = RandomizedSearchCV(pipe, m_p, n_iter=3, cv=3, scoring='roc_auc', verbose=1)
        search.fit(X_tr, y_tr)
        model = search.best_estimator_
        
        # Reports
        y_prd = model.predict(X_te)
        y_prb = model.predict_proba(X_te)[:, 1]
        print(f"\n🏆 TELCO REPORT ({model_type.upper()}):")
        print(classification_report(y_te, y_prd))
        print(f"Accuracy: {accuracy_score(y_te, y_prd):.2%}")
        print(f"ROC AUC: {roc_auc_score(y_te, y_prb):.4f}")
        
        # --- RESPONSIBLE AI: FAIRNESS AUDIT (Rubric 3.1.5) ---
        res = df_te.copy(); res['pred'] = y_prd
        audit = res.groupby('gender').agg(Actual=('churn', 'mean'), Predicted=('pred', 'mean'))
        print(f"⚖️ Gender Audit:\n{audit}")
        
        # Tính toán và Log Bias Gap lên MLflow
        bias_gap = abs(audit.loc['Female', 'Predicted'] - audit.loc['Male', 'Predicted'])
        mlflow.log_metric("bias_gap_gender", bias_gap)
        print(f"   - Logged Bias Gap: {bias_gap:.4f}")
        
        audit.to_csv("gender_audit.csv")
        mlflow.log_artifact("gender_audit.csv")
        
        # --- RESPONSIBLE AI: EXPLAINABILITY (SHAP) ---
        try:
            print("🔍 Generating SHAP summary plot...")
            X_t = model.named_steps['pre'].transform(X_te)
            if hasattr(X_t, "toarray"): X_t = X_t.toarray()
            
            explainer = shap.Explainer(model.named_steps['clf'])
            shap_v = explainer(X_t)
            
            import matplotlib.pyplot as plt
            plt.figure(figsize=(10, 6))
            shap.summary_plot(shap_v, X_t, feature_names=model.named_steps['pre'].get_feature_names_out(), show=False)
            plt.tight_layout()
            plt.savefig("shap_summary.png")
            mlflow.log_artifact("shap_summary.png") # Log biểu đồ lên MLflow
            plt.show()
        except Exception as e: 
            print(f"⚠️ SHAP failed: {e}")

        # Đăng ký model vào Registry
        mlflow.sklearn.log_model(model, "model", registered_model_name=f"CustomerChurnModel_{model_type}")
        print(f"✅ TELCO REGISTERED: {model_type.upper()}")

for m in ["xgboost", "logistic_regression"]:
    run_telco_experiment(df_train, df_test, m)
